In [ ]:
from reader import Reader
from encoder import *

models = {
    'bunny': Reader.read_from_file('assets/bunny.obj'),
    'eyeball': Reader.read_from_file('assets/eyeball.obj'),
    'torus': Reader.read_from_file('assets/torus.obj'),
}

In [ ]:
# Baseline
print('=== Baseline ===')
for name, model in models.items():
    m = model.copy()
    enc = BaselineEncoder()
    res = enc.encode(m)
    print(f'{name}: {res.bits_per_vertex:.2f} bpv, {len(res.data)} bytes')

In [ ]:
# Current best: PackedGTSEllipsoidFitter
print('=== PackedGTSEllipsoidFitter (current best) ===')
for name, model in models.items():
    m = model.copy()
    enc = PackedGTSEllipsoidFitter(
        vertex_quantization_error=0.0005,
        max_hierarchical_fitting_depth=3,
        verbose=True
    )
    res = enc.encode(m)
    print(f'{name}: {res.bits_per_vertex:.2f} bpv, {len(res.data)} bytes')
    print()

In [ ]:
# New: SphericalHierarchicalFitter with different tree depths
for depth in [2, 3, 4, 5, 6]:
    print(f'=== SphericalHierarchicalFitter (depth={depth}) ===')
    for name, model in models.items():
        m = model.copy()
        enc = SphericalHierarchicalFitter(
            spherical_tree_depth=depth,
            vertex_quantization_error=0.0005,
            max_hierarchical_fitting_depth=1,  # single ellipsoid, let spherical tree do the work
            verbose=True
        )
        res = enc.encode(m)
        print(f'{name}: {res.bits_per_vertex:.2f} bpv, {len(res.data)} bytes')
        print()
    print('---')

In [ ]:
# Detailed analysis for bunny with depth=4
print('=== Detailed: SphericalHierarchicalFitter bunny depth=4 ===')
m = models['bunny'].copy()
enc = SphericalHierarchicalFitter(
    spherical_tree_depth=4,
    vertex_quantization_error=0.0005,
    max_hierarchical_fitting_depth=1,
    verbose=True
)
res = enc.encode(m)
print(f'\nTotal: {res.bits_per_vertex:.2f} bpv, {len(res.data)} bytes')

# Compare with baseline
baseline = BaselineEncoder().encode(models['bunny'].copy())
print(f'Compression ratio vs baseline: {len(baseline.data) / len(res.data):.2f}x')